![QuantConnect Logo](https://cdn.quantconnect.com/web/i/icon.png)
<hr>

## Custom Indicator Research

This notebook implements a custom Money Flow Index and builds value and signal series from daily SPY history.

### Set Up QuantBook

Create a daily SPY subscription for the custom indicator updates.

In [ ]:
qb = QuantBook()
qb.set_start_date(2024, 12, 31)
qb.settings.seed_initial_prices = True
equity = qb.add_equity("SPY")

### Define Indicator

Implement a 20-period [custom Money Flow Index](https://www.quantconnect.com/docs/v2/writing-algorithms/indicators/custom-indicators).

In [ ]:
class CustomMoneyFlowIndex(PythonIndicator):

    def __init__(self, period: int) -> None:
        super().__init__()
        self.value = 0
        self._previous_typical_price = 0
        self._negative_money_flow: RollingWindow[float] = RollingWindow(period)
        self._positive_money_flow: RollingWindow[float] = RollingWindow(period)

    def update(self, input: TradeBar) -> bool:
        # Estimate the money flow by averaging the price multiplied by volume.
        typical_price = (input.high + input.low + input.close) / 3
        money_flow = typical_price * input.volume
        # Classify the flow as positive or negative relative to the previous bar.
        self._negative_money_flow.add(money_flow if typical_price < self._previous_typical_price else 0)
        self._positive_money_flow.add(money_flow if typical_price > self._previous_typical_price else 0)
        self._previous_typical_price = typical_price
        positive_money_flow_sum = sum(self._positive_money_flow)
        total_money_flow = positive_money_flow_sum + sum(self._negative_money_flow)
        # Set the value to the positive money flow ratio.
        self.value = 100
        if total_money_flow != 0:
            self.value *= positive_money_flow_sum / total_money_flow
        return self._positive_money_flow.is_ready

### Build Time Series

Feed daily TradeBar history through the indicator and store its value for each bar.

In [ ]:
mfi = CustomMoneyFlowIndex(20)
mfi_by_date = {
    bar.end_time: mfi.value
    for bar in qb.history[TradeBar](equity, 250, Resolution.DAILY)
    if mfi.update(bar)
}

indicator_values = pd.Series(mfi_by_date, name="mfi")
indicator_values

### Signal Series

Derive long and short signals based on the money flow indicator.

In [ ]:
# Go long when demand outweighs supply, otherwise go short.
signal = pd.Series(-1, index=indicator_values.index, name="signal")
signal[indicator_values > 50] = 1
signal